# Proof of concept

# LoRA experiments with Llava one vision

#### Jupyter kernel setup (uv)

1. Install Google Cloud SDK
   
2. Create the environment:
   ```bash
   uv venv .poc-lora
   ```

3. Activate it (terminal step; run before continuing):
   ```bash
   source .poc-lora/bin/activate
   ```

4. Install notebook + model dependencies:
   ```bash
   uv pip install jupyter ipykernel "transformers>=4.41" accelerate safetensors pillow pandas datasets gcsfs
   ```

5. Register the kernel:
   ```bash
   python -m ipykernel install --user --name=poc-lora --display-name="POC LoRA"
   ```

6. In `experiments/poc.ipynb`, pick the `Where You LoRA (vision)` kernel before running any cells.

In [ ]:
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration
import torch
import pandas as pd
from datasets import load_dataset

### Constants

In [ ]:
target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",  # attention
    "up_proj", "gate_proj", "down_proj",    # MLP
    "vision_resampler.dense",              # vision adapter
]

### Functions

### Datasets

In [ ]:
# VQAv2 Dataset
ds = load_dataset("lmms-lab/VQAv2")

In [ ]:
#GQA Dataset
GQA_all = load_dataset("lmms-lab/GQA", "challenge_all_images")
GQA_all_instruct = load_dataset("lmms-lab/GQA", "challenge_all_instructions")
GQA_balanced = load_dataset("lmms-lab/GQA", "challenge_balanced_images")

In [ ]:
### HallusionBench Dataset

splits = {'image': 'data/image-00000-of-00001.parquet', 'non_image': 'data/non_image-00000-of-00001.parquet'}
hallusionbench = pd.read_parquet("hf://datasets/lmms-lab/HallusionBench/" + splits["image"])

In [ ]:
#POPE
pope = load_dataset("lmms-lab/POPE", "Full")

### Model Set Up

In [ ]:
model_id = "llava-hf/LLaVA-OneVision-Qwen2-72B-OV"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=dtype,
    device_map="auto",
)

image = processor.image_proc(Image.open("your_image.jpg"))
prompt = processor.apply_chat_template(
    [{"role": "user", "content": [{"type": "text", "text": "Describe the image"}, {"type": "image"}]}],
    add_generation_prompt=True,
    return_tensors="pt",
)

inputs = processor(image, prompt, return_tensors="pt").to(model.device, dtype)
output = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(output[0], skip_special_tokens=True))

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()